# GRAD-N: Real Network Import — TX-123BT, then TAMU ACTIVSg
## REE 4301 / IE 5300 / IE 6301 — Energy Systems Modeling
### Graduate sections only. Individual work. After Mini-Project 3.

**The problem this assignment exists to solve.** Every network you have built in this course has between three and six nodes. You have reported binding constraints, congestion rents, and locational marginal prices from those models. None of them has ever been checked against a network that looks anything like the real thing.

Here you import **TX-123BT** — a synthetic 123-bus, 345 kV backbone that mirrors ERCOT's spatial and temporal characteristics, with five years of weather-driven hourly profiles — solve a DC optimal power flow on it, and compare what it says to what your own aggregated model said.

**What you will learn:**
- How to get a real network dataset into PyPSA, including the parts that do not import cleanly
- Why *validating* data you did not create is most of the work — this dataset has at least three defects you will find in Part 1
- How much of your Mini-Project 3 answer was an artefact of aggregation
- That spatial resolution is a modelling choice with a cost, not a detail

**Stage 1 (Parts 0–5) is required. Stage 2 (Part 6) is an optional extension.**

---
### The dataset

Jin Lu, Xingpeng Li, Hongyi Li, Taher Chegini, Carlos Gamarra, Y. C. Ethan Yang, Margaret Cook, and Gavin Dillingham, *A Synthetic Texas Backbone Power System with Climate-Dependent Spatio-Temporal Correlated Profiles*. Dataset: DOI [10.6084/m9.figshare.22144616](https://doi.org/10.6084/m9.figshare.22144616), CC BY 4.0. Landing page: <https://rpglab.github.io/resources/TX-123BT/>.

**Cite it in your report.** The licence requires attribution.


---
## Part 0 — Setup and data acquisition

The complete TX-123BT release is a **544 MB ZIP** — five years of hourly climate, load, solar, wind and line-rating profiles. You need four small files out of it (about 67 KB total) plus one day of profiles.

Rather than download half a gigabyte, the cell below reads the archive's central directory over HTTP **range requests** and pulls out only the members it needs. This is worth knowing as a technique in its own right: it works on any static ZIP served by a host that honours `Range:` headers, and it turns a 544 MB download into about 1.3 MB.


In [ ]:
!pip install -q pypsa highspy openpyxl


---

### Choosing a solver



This notebook defaults to **HiGHS**, which is open source, needs no licence, and has no size limit — because every model in it is well past what Gurobi's free licence allows. Gurobi is still the faster option if you have an academic key; see below.



That ceiling arrives sooner than you would think. Measured sizes for the models in this course:



| model | variables | constraints | restricted licence |

|---|---|---|---|

| 1-node, 24 hours | 123 | 291 | fits |

| SB6 Stage 1 | 766 | 1,837 | fits |

| SB6 Stage 2 | 886 | 2,172 | too big |

| SB6 Stage 3 | 2,614 | 6,444 | too big |

| TX-123BT, 24 hours | 16,080 | 38,304 | 19x over |



When you exceed it, Gurobi returns *"Model too large for size-limited license"*. Two ways past it:



**In class — switch to HiGHS.** Open source, no licence, no size limit. Uncomment the `SOLVER` line below. There is no accuracy cost: HiGHS and Gurobi agree to sixteen significant figures on every model here. The speed cost is real only when the problem is large — measured on a 1-node model, HiGHS is *marginally faster* at 24 hours, identical at one week, and about **12x slower** on a full 8,760-hour year (22 s against 1.8 s). On a mixed-integer unit-commitment problem with 2,880 binary variables the gap was only **1.8x**.



**For homework — get an academic licence.** A free Web License Service (WLS) key from gurobi.com works in Colab with **no licence file**: paste the three values into `WLS` below. Build the environment **once** and reuse it — constructing a new one for every solve re-authenticates each time and will exhaust a WLS session partway through a scenario sweep.



In [ ]:
SOLVER = 'highs'      # this notebook's models are far over the

                      # restricted-licence ceiling, so HiGHS is

                      # the default here

# SOLVER = 'gurobi'   # <-- faster, but needs a WLS key for models this size



# For homework: paste your academic Web License Service key here.

# Leave it empty and Gurobi falls back to its restricted licence.

WLS = {}   # {'WLSACCESSID': '...', 'WLSSECRET': '...', 'LICENSEID': 000000}



ENV = None

if SOLVER == 'gurobi' and WLS:

    import gurobipy as gp

    ENV = gp.Env(params=WLS)     # ONE environment, reused by every solve



print(f'solver: {SOLVER}'

      + ('  (academic WLS licence)' if ENV else '  (default licence)'))



In [ ]:
import io
import zipfile

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)


class HttpFile(io.RawIOBase):
    """A seekable read-only file over HTTP range requests.

    Enough of the file protocol for `zipfile` to work with: it reads the
    end-of-central-directory record, then only the members you ask for.
    """

    def __init__(self, url, session=None):
        self.url = url
        self.s = session or requests.Session()
        self.pos = 0
        self.n_requests = 0
        self.n_bytes = 0
        r = self.s.get(url, headers={'Range': 'bytes=0-0'},
                       timeout=60, stream=True)
        r.raise_for_status()
        if 'Content-Range' not in r.headers:
            raise RuntimeError('server does not support range requests')
        self.size = int(r.headers['Content-Range'].split('/')[1])
        r.close()

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = (offset if whence == 0 else
                    self.pos + offset if whence == 1 else
                    self.size + offset)
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        if n == 0 or self.pos >= self.size:
            return b''
        end = min(self.pos + n, self.size) - 1
        r = self.s.get(self.url,
                       headers={'Range': f'bytes={self.pos}-{end}'},
                       timeout=120)
        r.raise_for_status()
        data = r.content
        self.pos += len(data)
        self.n_requests += 1
        self.n_bytes += len(data)
        return data


TX123_ZIP = 'https://ndownloader.figshare.com/files/44942761'
remote = HttpFile(TX123_ZIP)
archive = zipfile.ZipFile(remote)
print(f'archive: {remote.size/1e6:.1f} MB, {len(archive.namelist()):,} members')


### Choose your day

The profiles cover 2017–2021. The default below is **day 46 of 2021 — 15 February 2021**, the second day of Winter Storm Uri, because it is the most interesting 24 hours in the dataset and it connects directly to Chapter 15 of the book.

You may pick a different day. If you do, say which and why in your report — the day you choose changes every number you will report.


In [ ]:
YEAR = 2021
DAY = 46          # day-of-year, 1-365.  46 = 15 Feb 2021 (Winter Storm Uri)

P = 'Data_public_5year/'
WANT = {
    'bus':    P + 'Bus_data.csv',
    'line':   P + 'Line_data.csv',
    'gen':    P + 'Generator_data.xlsx',
    'readme': P + 'Readme.txt',
    'load':   P + f'Load_5y/Load_annual_{YEAR}/load_annual_D{DAY}.txt',
    'solar':  P + f'Solar_5y/solar_{YEAR}/solar_annual_D{DAY}.txt',
    'wind':   P + f'Wind_5y/wind_{YEAR}/wind_annual_D{DAY}.txt',
    'rating': P + f'Daily_line_rating_5y/line_annual_{YEAR}.txt',
}

raw = {}
for key, member in WANT.items():
    raw[key] = archive.read(member)
    print(f'  {len(raw[key]):>9,} bytes  {member}')

print(f'\ndownloaded {remote.n_bytes/1024:,.0f} KB in {remote.n_requests} range requests '
      f'instead of {remote.size/1e6:.0f} MB')


---
## Part 1 — Validate before you trust

You did not create this data and the people who did are not available to answer questions. Before any of it goes into a model, check that it says what its column headers claim it says.

This is not a formality. **There are at least three defects in the files you just downloaded.** Two of them are found below; the third is left for you in the exercise at the end of this Part.


In [ ]:
bus = pd.read_csv(io.BytesIO(raw['bus']))
line = pd.read_csv(io.BytesIO(raw['line']))
gen = pd.read_excel(io.BytesIO(raw['gen']), sheet_name='Gen data')
solar_map = pd.read_excel(io.BytesIO(raw['gen']),
                          sheet_name='Solar Plant Number')
wind_map = pd.read_excel(io.BytesIO(raw['gen']),
                         sheet_name='Wind Plant Number')

load = np.loadtxt(io.BytesIO(raw['load']))
solar = np.loadtxt(io.BytesIO(raw['solar']))
wind = np.loadtxt(io.BytesIO(raw['wind']))
daily_rating = np.loadtxt(io.BytesIO(raw['rating']))

print('bus  ', bus.shape, list(bus.columns))
print('line ', line.shape)
print('gen  ', gen.shape, list(gen.columns))
print()
print('load  ', load.shape)
print('solar ', solar.shape)
print('wind  ', wind.shape)
print('rating', daily_rating.shape, '(lines x days)')


### Defect 1 — the profile arrays are not oriented the same way

Look at the three shapes above. The load file is **(24 hours × 123 buses)**. The solar and wind files are **(plants × 24 hours)** — transposed relative to the load file, in the same release, documented correctly in `Readme.txt` and easy to miss.

Get this wrong and nothing raises an error; you simply model a different system. Assert the orientation rather than assuming it.


In [ ]:
N_HOURS = 24
assert load.shape == (N_HOURS, len(bus)), \
    f'expected load as (hours, buses), got {load.shape}'
assert solar.shape == (len(solar_map), N_HOURS), \
    f'expected solar as (plants, hours), got {solar.shape}'
assert wind.shape == (len(wind_map), N_HOURS), \
    f'expected wind as (plants, hours), got {wind.shape}'

system_load = load.sum(axis=1)
print(f'system load: min {system_load.min():,.0f} MW  '
      f'peak {system_load.max():,.0f} MW  '
      f'energy {load.sum():,.0f} MWh')
print(f'wind fleet:  {wind.sum(axis=0).min():,.0f} - {wind.sum(axis=0).max():,.0f} MW')
print(f'solar fleet: peak {solar.sum(axis=0).max():,.0f} MW')


### Defect 2 — `Line_data.csv`'s coordinate columns are mislabelled

The header reads `From Bus Latitude, From Bus Longitude, To Bus Latitude, To Bus Longitude`. The values are ordered **From-latitude, To-latitude, From-longitude, To-longitude**: columns 8 and 9 are swapped relative to their labels.

You can prove this rather than guess it, because `Bus_data.csv` gives the true coordinate of every bus. Check each candidate mapping against it and keep the one with zero error.


In [ ]:
coords = bus.set_index('Bus Number')
truth = {
    'from-lat': coords['Bus latitude'].loc[line['From Bus Number']].values,
    'to-lat':   coords['Bus latitude'].loc[line['To Bus Number']].values,
    'from-lon': coords['Bus longitude'].loc[line['From Bus Number']].values,
    'to-lon':   coords['Bus longitude'].loc[line['To Bus Number']].values,
}
cols = list(line.columns)
claimed = {7: 'from-lat', 8: 'from-lon', 9: 'to-lat', 10: 'to-lon'}

print('mean absolute error against the true bus coordinates, in degrees:')
print(f"{'col':>4}  {'header says':<22}{'error if believed':>18}  {'best match':<9}{'error':>9}")
for j, says in claimed.items():
    v = line[cols[j]].values
    errs = {k: float(np.abs(v - t).mean()) for k, t in truth.items()}
    best = min(errs, key=errs.get)
    flag = '' if best == says else '   <-- MISLABELLED'
    print(f'{j:>4}  {cols[j]:<22}{errs[says]:>18.4f}  '
          f'{best:<9}{errs[best]:>9.4f}{flag}')

print('\nColumns 8 and 9 are swapped relative to their headers. '
      'True order: From-lat, To-lat, From-lon, To-lon.')
line = line.rename(columns={
    cols[7]: 'from_lat', cols[8]: 'to_lat',
    cols[9]: 'from_lon', cols[10]: 'to_lon'})


### The sanity check that catches this without any arithmetic

Plot the network. If the coordinates are wired up correctly it looks like Texas; if they are not, it looks like nothing. **Plot every network you import, before you solve it.**


In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 8))
for _, r in line.iterrows():
    ax.plot([r['from_lon'], r['to_lon']], [r['from_lat'], r['to_lat']],
            color='0.7', lw=0.8, zorder=1)
gen_bus = set(gen['Bus Number'])
is_gen = bus['Bus Number'].isin(gen_bus)
ax.scatter(bus.loc[~is_gen, 'Bus longitude'], bus.loc[~is_gen, 'Bus latitude'],
           s=14, color='#0064B1', zorder=2, label='load-only bus')
ax.scatter(bus.loc[is_gen, 'Bus longitude'], bus.loc[is_gen, 'Bus latitude'],
           s=26, color='#DE6B1A', zorder=3, label='generator bus')
ax.set_title(f'TX-123BT: {len(bus)} buses, {len(line)} lines, 345 kV backbone')
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.legend(frameon=False); ax.set_aspect(1.15)
plt.tight_layout(); plt.show()


### The generator table

Two things to notice, one of which is the third defect.

The **fuel mix** should look like ERCOT: gas-dominated, a lot of wind, a little coal and nuclear. Check that it does.

The **cost columns** are `C0($/MWh)` and `C1($/MWh)`. Both are labelled per-MWh. Look at the nuclear unit: `C0 = 0`, `C1 = 17.44`. Now look at a gas unit: `C0 = 597` on a 305 MW machine. A no-load cost of $597/MWh is not credible; a no-load cost of **$597 per hour** is. `C0` is a $/h no-load cost with a wrong unit label, and `C1` is the marginal cost you want. Using `C0` as an energy cost would inflate dispatch cost by two orders of magnitude on some units.


In [ ]:
mix = gen.groupby('Fuel type')['Pmax (MW)'].agg(['count', 'sum'])
mix['share'] = 100 * mix['sum'] / mix['sum'].sum()
print(mix.round(1).to_string())
print(f"\ntotal installed {gen['Pmax (MW)'].sum():,.0f} MW "
      f'against a {system_load.max():,.0f} MW peak')
print()
print(gen.loc[gen['Fuel type'].isin(['Nuclear', 'Natural Gas']),
             ['Gen Number', 'Fuel type', 'Pmax (MW)', 'C0($/MWh)',
              'C1($/MWh)', 'Csu($)']].head(6).to_string(index=False))


> **Exercise 1.1 — find the third defect.** One of the two remaining quantities you are about to model has a documented meaning that does not match how a first reading of the file would suggest using it. Compare the static `Capacity (MW)` column in `Line_data.csv` against the daily values in `daily_rating` for your chosen day. State what the difference is, which one the dataset's own sample SCUC code uses, and why a **cold** day pushes the number in the direction it does. Part 4 turns this into an experiment; answer it here first, from the data.


---
## Part 2 — Build the network in PyPSA

Three unit conversions matter, and none of them is checked for you.

**Impedance.** `R, pu` and `X, pu` are per-unit on the system base. PyPSA wants ohms. With a 100 MVA base at 345 kV the base impedance is 345² / 100 = 1,190.25 Ω, so `x_ohm = x_pu × 1190.25`. Get this wrong by a constant factor and the *relative* impedances stay right, so the flows look plausible and are wrong.

**Renewable output.** The solar and wind files give **MW produced**, not a capacity factor. PyPSA's `p_max_pu` is a fraction of `p_nom`, so divide by the plant's `Pmax` and clip to [0, 1].

**Commitment.** Many units have `Pmin > 0`. That is a unit-commitment constraint and needs binary variables. This notebook solves the linear relaxation (`p_min_pu = 0`), which is what makes it run in seconds instead of minutes. **Say so in your report** — it is the single largest difference between this model and the SCUC the dataset ships.


In [ ]:
import pypsa

BASE_MVA = 100.0
V_NOM = 345.0
Z_BASE = V_NOM ** 2 / BASE_MVA          # 1190.25 ohm
VOLL = 9000.0                           # $/MWh value of lost load


def build_tx123(line_rating=None, voll=VOLL):
    """Assemble TX-123BT as a PyPSA network for one 24-hour day.

    line_rating : None -> use the static `Capacity (MW)` column
                  array  -> per-line MW rating to use instead
    voll        : add a load-shedding generator at every bus at this price.
                  Always do this on an imported network.  Without it an
                  over-constrained hour returns `infeasible` and tells you
                  nothing; with it you get MWh unserved, and where.
    """
    n = pypsa.Network(name='TX-123BT')
    n.set_snapshots(pd.RangeIndex(N_HOURS, name='hour'))
    n.add('Carrier', 'AC')

    n.add('Bus', ('B' + bus['Bus Number'].astype(str)).values,
          v_nom=V_NOM, carrier='AC',
          x=bus['Bus longitude'].values, y=bus['Bus latitude'].values)

    s_nom = (line['Capacity (MW)'].values if line_rating is None
             else np.asarray(line_rating, dtype=float))
    n.add('Line', ('L' + line['line_num'].astype(str)).values,
          bus0=('B' + line['From Bus Number'].astype(str)).values,
          bus1=('B' + line['To Bus Number'].astype(str)).values,
          carrier='AC',
          r=line['R, pu'].values * Z_BASE,
          x=line['X, pu'].values * Z_BASE,
          s_nom=s_nom,
          length=line['Length (Mile)'].values * 1.60934)

    for fuel in gen['Fuel type'].unique():
        n.add('Carrier', fuel)
    solar_of = dict(zip(solar_map['Generator Number'],
                        solar_map['Solar Plant Number']))
    wind_of = dict(zip(wind_map['Generator Number'],
                       wind_map['Wind Plant Number']))

    for _, g in gen.iterrows():
        gid = int(g['Gen Number'])
        pmax = float(g['Pmax (MW)'])
        kw = dict(bus=f"B{int(g['Bus Number'])}", carrier=g['Fuel type'],
                  p_nom=pmax,
                  marginal_cost=float(g['C1($/MWh)']),   # NOT C0
                  p_min_pu=0.0)                          # LP relaxation
        if gid in solar_of and pmax > 0:
            kw['p_max_pu'] = pd.Series(
                np.clip(solar[solar_of[gid] - 1] / pmax, 0, 1),
                index=n.snapshots)
        elif gid in wind_of and pmax > 0:
            kw['p_max_pu'] = pd.Series(
                np.clip(wind[wind_of[gid] - 1] / pmax, 0, 1),
                index=n.snapshots)
        n.add('Generator',
              f"G{gid}_{g['Fuel type'].replace(' ', '')}", **kw)

    if voll is not None:
        n.add('Carrier', 'load shedding')
    for j, b in enumerate(bus['Bus Number']):
        n.add('Load', f'D{b}', bus=f'B{b}',
              p_set=pd.Series(load[:, j], index=n.snapshots))
        if voll is not None:
            n.add('Generator', f'SHED{b}', bus=f'B{b}',
                  carrier='load shedding',
                  p_nom=float(load[:, j].max()) * 1.5 + 1.0,
                  marginal_cost=voll)
    return n


n = build_tx123()
print(f'{len(n.buses)} buses | {len(n.lines)} lines | '
      f'{len(n.generators)} generators | {len(n.loads)} loads | '
      f'{len(n.snapshots)} snapshots')


---
## Part 3 — Solve the DC optimal power flow

PyPSA writes the linearised power-flow constraints of Chapter 18 — Kirchhoff's voltage law around every independent loop, plus a nodal balance at every bus — and minimises dispatch cost subject to line thermal limits.


In [ ]:
status, condition = n.optimize(solver_name=SOLVER, env=ENV)
print(status, condition)


In [ ]:
def summarise(n, label=''):
    """Report the four things worth reporting from a solved network."""
    shed_cols = [c for c in n.generators_t.p.columns
                 if c.startswith('SHED')]
    shed = n.generators_t.p[shed_cols]
    unserved = shed.to_numpy().sum()
    lmp = n.buses_t.marginal_price
    util = n.lines_t.p0.abs().div(n.lines.s_nom, axis=1)
    binding = util.max()[util.max() > 0.999].sort_values(ascending=False)
    print(f'--- {label}')
    print(f'    daily cost        ${n.objective:,.0f}')
    print(f'    unserved energy   {unserved:,.1f} MWh')
    print(f'    LMP               ${lmp.to_numpy().min():,.2f} .. '
          f'${lmp.to_numpy().max():,.2f} /MWh '
          f'(mean ${lmp.to_numpy().mean():,.2f})')
    print(f'    binding lines     {len(binding)}')
    return binding, unserved


binding, unserved = summarise(n, 
                              f'static Capacity column, day {DAY} of {YEAR}')


In [ ]:
names = bus.set_index('Bus Number')['Bus Name']


def congestion_table(n, top=8):
    """Which corridors bind, for how many hours, and between where."""
    util = n.lines_t.p0.abs().div(n.lines.s_nom, axis=1)
    rows = []
    for ln in util.max().sort_values(ascending=False).head(top).index:
        b0 = int(n.lines.at[ln, 'bus0'][1:])
        b1 = int(n.lines.at[ln, 'bus1'][1:])
        rows.append({
            'line': ln,
            'from': names[b0], 'to': names[b1],
            'rating MW': round(n.lines.at[ln, 's_nom'], 0),
            'peak use %': round(100 * util[ln].max(), 1),
            'hours binding': int((util[ln] > 0.999).sum()),
        })
    return pd.DataFrame(rows)


print(congestion_table(n).to_string(index=False))


In [ ]:
gen_mix = (n.generators_t.p.T.groupby(n.generators.carrier).sum().sum(axis=1)
           .sort_values(ascending=False))
gen_mix = gen_mix[gen_mix > 0.5]
print('generation over the day (MWh):')
for carrier, mwh in gen_mix.items():
    print(f'  {carrier:16s} {mwh:12,.0f}   {100*mwh/gen_mix.sum():5.1f}%')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(n.generators_t.p.T.groupby(n.generators.carrier).sum().T
 .loc[:, gen_mix.index].plot.area(ax=ax[0], lw=0, alpha=0.85))
n.loads_t.p_set.sum(axis=1).plot(ax=ax[0], color='k', lw=1.6, label='load')
ax[0].set_title('Dispatch by carrier'); ax[0].set_ylabel('MW')
ax[0].set_xlabel('hour'); ax[0].legend(fontsize=8, ncol=2)

lmp = n.buses_t.marginal_price
ax[1].fill_between(lmp.index, lmp.min(axis=1), lmp.max(axis=1),
                   alpha=0.3, color='#0064B1', label='min-max across buses')
ax[1].plot(lmp.index, lmp.mean(axis=1), color='#0064B1', lw=2, label='mean')
ax[1].set_title('Locational marginal price'); ax[1].set_ylabel('$/MWh')
ax[1].set_xlabel('hour'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()


> **Exercise 3.1.** The LMP band above is the spatial price spread the network creates. In your Mini-Project 3 or SB6 model, how many distinct prices could there possibly be? Here there are up to 123 per hour. State what your aggregated model can and cannot say about locational value as a result.


---
## Part 4 — The line-rating experiment

This is the part of the assignment that is really about data assumptions rather than about Texas.

`Line_data.csv` gives one static `Capacity (MW)` per line. The release *also* ships `Daily_line_rating_5y`, a rating for every line on every day, computed from the weather — a **dynamic line rating**. Conductors sag when hot and can carry more current when the air is cold, so on a February morning in Texas the dynamic rating is well above the static one.

Solve the same day twice and compare.


In [ ]:
dlr = daily_rating[:, DAY - 1]
static = line['Capacity (MW)'].values
print(f'static  rating: {static.min():,.0f} / {np.median(static):,.0f} / {static.max():,.0f} MW  (min/median/max)')
print(f'dynamic rating: {dlr.min():,.0f} / {np.median(dlr):,.0f} / {dlr.max():,.0f} MW')
print(f'dynamic is {np.median(dlr/static):.2f}x the static rating on the median line')

results = {}
for label, rating in [('static Capacity column', None),
                      (f'dynamic rating, day {DAY}', dlr)]:
    m = build_tx123(line_rating=rating)
    m.optimize(solver_name=SOLVER, env=ENV, log_to_console=False)
    results[label] = m
    b, u = summarise(m, label)
    print()


In [ ]:
a, b = results.values()
la, lb = results.keys()
shed_a = a.generators_t.p[[c for c in a.generators_t.p.columns
                           if c.startswith('SHED')]].to_numpy().sum()
shed_b = b.generators_t.p[[c for c in b.generators_t.p.columns
                           if c.startswith('SHED')]].to_numpy().sum()
print(f'{la:28s} ${a.objective:>14,.0f}   {shed_a:>8,.1f} MWh unserved')
print(f'{lb:28s} ${b.objective:>14,.0f}   {shed_b:>8,.1f} MWh unserved')
print(f'{"difference":28s} ${a.objective-b.objective:>14,.0f}   '
      f'{shed_a-shed_b:>8,.1f} MWh')

print('\nwhere the unserved energy lands under the static rating:')
s = a.generators_t.p[[c for c in a.generators_t.p.columns
                      if c.startswith('SHED')]].sum()
s = s[s > 0.01].sort_values(ascending=False)
for k, v in s.items():
    print(f'  {names[int(k[4:])]:32s} {v:8,.1f} MWh')


> **Exercise 4.1.** Two runs of the same network on the same day, differing only in which line-rating column you believe, give materially different answers — a different daily cost, a different number of binding corridors, and in one case load shed that the other does not have.
>
> Neither column is wrong. State which one you would use for **(a)** a long-run capacity-expansion study and **(b)** a next-day operational study, and defend each choice in two sentences. Then say what this implies about reporting a single congestion result without stating the rating assumption behind it.

> **Exercise 4.2 — the connection to Chapter 15.** Your chosen day is in the middle of Winter Storm Uri. Compare the peak system load in this dataset for your day against a normal winter day (try day 20). Then answer: the real February 2021 failure was driven by generators freezing and by plants that could not get gas. **Can the model you just solved represent either of those?** Be specific about which component would have to carry the constraint.


---
## Part 5 — Compare against your own aggregated model

This is the deliverable. Everything above was setup.

Take the network you built for Mini-Project 3 or SB6 — three to six nodes, your own line capacities, your own demand — and put its results beside these. The comparison is not about which is *right*: TX-123BT is synthetic too. It is about which questions each one can answer.


In [ ]:
# Fill these in from your own Mini-Project 3 / SB6 run.
MY_MODEL = {
    'nodes': None,               # e.g. 4
    'lines': None,
    'binding_corridor': None,    # e.g. 'West Texas -> Dallas'
    'lmp_spread': None,          # max - min LMP in $/MWh
    'daily_cost': None,          # $ for a comparable day
}

ref = results[f'dynamic rating, day {DAY}']
ref_lmp = ref.buses_t.marginal_price
comparison = pd.DataFrame({
    'your model': [MY_MODEL['nodes'], MY_MODEL['lines'],
                   MY_MODEL['binding_corridor'], MY_MODEL['lmp_spread'],
                   MY_MODEL['daily_cost']],
    'TX-123BT': [len(ref.buses), len(ref.lines),
                 congestion_table(ref, top=1)['line'].iat[0],
                 round((ref_lmp.max(axis=1) - ref_lmp.min(axis=1)).max(), 2),
                 round(ref.objective, 0)],
}, index=['nodes', 'lines', 'top binding corridor',
          'max LMP spread ($/MWh)', 'daily cost ($)'])
print(comparison.to_string())


### What to hand in for Stage 1

1. **The import, working.** The network plot from Part 1 and the congestion table from Part 3, for a day you chose and justified.
2. **The three data defects.** Two are found in Part 1; the third is Exercise 1.1. For each, say how you detected it and what would have happened had you not.
3. **The rating experiment.** The two-run comparison from Part 4 with Exercise 4.1 answered.
4. **Two ways your own aggregated model over- or under-states the real system's constraints, and why.** This is the graded core of the assignment. Be specific — name the corridor, the price, or the constraint. "It has fewer nodes" is not an answer; "my model reports a single system price, so it cannot show the $X/MWh separation that appears between the Houston buses and West Texas in hours 18–21" is.
5. **The commitment relaxation.** State that you solved the LP relaxation rather than a unit-commitment problem, and give one result you would expect to move if you had not.


---
## Part 6 — Stage 2 (optional): the TAMU ACTIVSg cases

**Optional extension. Not required.** Flag it to the instructor before attempting a full multi-period run at this scale; a single-snapshot DC OPF is the realistic ceiling for a course-length assignment.

**Which case to use.** The catalog names ACTIVSg500. Note that **ACTIVSg500 is a synthetic *South Carolina* system** — the case header says so. The synthetic **Texas** case is **ACTIVSg2000**. If your point is the like-for-like comparison against TX-123BT, use ACTIVSg2000. If your point is only what changes with scale and solve time, ACTIVSg500 is smaller and faster, but say in your report that you have changed geography as well as size.

Both ship as MATPOWER `.m` files in the MATPOWER repository, which is a cleaner fetch than the TAMU site (which requires a registration click).


In [ ]:
import re


def parse_matpower(text):
    """Minimal MATPOWER .m -> PYPOWER ppc dict."""
    ppc = {'version': '2'}
    m = re.search(r'mpc\.baseMVA\s*=\s*([0-9.eE+-]+)\s*;', text)
    ppc['baseMVA'] = float(m.group(1)) if m else 100.0
    for field in ('bus', 'gen', 'branch', 'gencost'):
        m = re.search(r'mpc\.%s\s*=\s*\[(.*?)\n\s*\];' % field,
                      text, re.S)
        if not m:
            continue
        rows = []
        for raw_line in m.group(1).splitlines():
            s = raw_line.split('%')[0].strip().rstrip(';').strip()
            if s:
                rows.append([float(x) for x in
                             s.replace(',', ' ').split()])
        w = max(len(r) for r in rows)
        ppc[field] = np.array([r + [0.0] * (w - len(r)) for r in rows])

    # Cell-array fields:  mpc.genfuel = { 'ng'; 'wind'; ... };
    # BOTH standard converters drop these, and they are the only place the
    # case says what anything burns.  Without them you have 544 anonymous
    # machines and cannot report a generation mix at all.
    for field in ('gentype', 'genfuel', 'bus_name'):
        m = re.search(r'mpc\.%s\s*=\s*\{(.*?)\};' % field, text, re.S)
        if m:
            ppc[field] = re.findall(r"'([^']*)'", m.group(1))
    return ppc


def marginal_cost_from_gencost(ppc):
    """MATPOWER gencost -> $/MWh.

    `import_from_pypower_ppc` does NOT carry gencost across, so without
    this the objective is empty and PyPSA refuses to build a model.

    STATED APPROXIMATION: model 2 is a quadratic c2*p^2 + c1*p + c0, and a
    PyPSA LP takes one constant marginal cost per generator, so this keeps
    c1 and discards the curvature.  On ACTIVSg2000 that understates the
    marginal cost at full output by a mean of $0.38/MWh (median $0.21, max
    $4.03); 50 of 544 units are off by more than $1/MWh.  Small, but it is
    an approximation and you should say so.  See Exercise 6.1.
    """
    out = []
    for row in ppc['gencost']:
        model, ncost = int(row[0]), int(row[3])
        c = row[4:4 + ncost]
        if model == 2:                    # polynomial, descending powers
            out.append(float(c[-2]) if ncost >= 2 else 0.0)
        else:                             # piecewise linear x1,y1,x2,y2,...
            pts = c.reshape(-1, 2)
            dx = pts[1, 0] - pts[0, 0]
            out.append(float((pts[1, 1] - pts[0, 1]) / dx) if dx else 0.0)
    return out


# MATPOWER fuel codes -> readable carrier names
FUEL_NAMES = {'ng': 'natural gas', 'coal': 'coal', 'nuclear': 'nuclear',
              'hydro': 'hydro', 'wind': 'wind', 'solar': 'solar',
              'oil': 'oil', 'biomass': 'biomass',
              'geothermal': 'geothermal', 'other': 'other'}


CASE = 'case_ACTIVSg2000'      # Texas.  'case_ACTIVSg500' is South Carolina.
url = ('https://raw.githubusercontent.com/MATPOWER/matpower/master/data/'
       f'{CASE}.m')
text = requests.get(url, timeout=120).text
ppc = parse_matpower(text)
print(CASE)
print('  numeric tables:',
      {k: v.shape for k, v in ppc.items() if hasattr(v, 'shape')})
print('  cell arrays:  ',
      {k: len(ppc[k]) for k in ('gentype', 'genfuel', 'bus_name')
       if k in ppc})
print(' ', text.splitlines()[0])


### The three things that make this import fail silently

`import_from_pypower_ppc` is a convenience, not a complete translation. Two of its gaps produce the same unhelpful symptom — `infeasible`, with no message saying why — and the third produces something worse, which is a result that looks fine and is missing half the information.

1. **`gencost` is not imported.** No costs, so no objective, and PyPSA raises before it solves.
2. **`Pg` — MATPOWER's own *solved dispatch* — is imported into the static `generators.p_set` column.** PyPSA then writes a `Generator-p_set` equality constraint pinning every generator to that value. The problem is over-determined: 544 generation levels are fixed, so the nodal balance cannot also hold, and the solver returns infeasible. **You are not optimising anything.** Setting `p_set` to `NaN` releases it.
3. **`mpc.genfuel`, `mpc.gentype` and `mpc.bus_name` are dropped entirely** — by `import_from_pypower_ppc` and by pandapower's converter alike, because they are MATLAB cell arrays rather than numeric matrices. They are the only place the case records what anything burns or what anything is called. Lose them and you have 544 anonymous machines: the model still solves, the dispatch is still right, and you cannot report a generation mix, tell wind from gas, or name a single congested corridor. The parser above reads them; the cell below attaches them.

If you take one habit away from this assignment, make it this: after any network import, look at what constraints the model actually contains (`n.optimize.create_model().constraints`) before believing a result — or before believing an infeasibility. And check what the source file held that your object no longer does.


In [ ]:
m = pypsa.Network()
m.import_from_pypower_ppc(ppc)

print('as imported:')
print(f'  {len(m.buses)} buses, {len(m.lines)} lines, '
      f'{len(m.transformers)} transformers, {len(m.generators)} generators, '
      f'{len(m.loads)} loads')
print(f'  generators.p_set is set on {int(m.generators.p_set.notna().sum())}'
      f' of {len(m.generators)} generators, summing to '
      f'{m.generators.p_set.sum():,.0f} MW against '
      f'{m.loads.p_set.sum():,.0f} MW of load')

# look at what the model would contain before fixing anything
m.generators['marginal_cost'] = marginal_cost_from_gencost(ppc)
print('\nconstraints PyPSA would build:')
for name, con in m.optimize.create_model().constraints.items():
    flag = '   <-- this one pins every generator' if 'p_set' in name else ''
    print(f'  {name:30s} {con.shape}{flag}')


In [ ]:
m.generators['p_set'] = np.nan     # release MATPOWER's solved dispatch
m.generators['p_min_pu'] = 0.0     # LP relaxation, as in Part 2

# Put back what the converter dropped.
carriers = [FUEL_NAMES.get(f, f) for f in ppc['genfuel']]
for carrier in sorted(set(carriers)):
    m.add('Carrier', carrier)
m.generators['carrier'] = carriers
m.buses['substation'] = list(ppc['bus_name'])

fleet = (m.generators.groupby('carrier').p_nom
         .agg(['count', 'sum']).sort_values('sum', ascending=False))
print('installed capacity by fuel:')
for carrier, r in fleet.iterrows():
    print(f"  {carrier:14s} {int(r['count']):4d} units  {r['sum']:10,.0f} MW"
          f"  {100*r['sum']/fleet['sum'].sum():5.1f}%")


In [ ]:
status, condition = m.optimize(solver_name=SOLVER, env=ENV)
print(status, condition)
if condition == 'optimal':
    lmp = m.buses_t.marginal_price
    util = m.lines_t.p0.abs().div(m.lines.s_nom, axis=1).iloc[0]
    print(f'  hourly cost   ${m.objective:,.0f}')
    print(f'  LMP           ${lmp.to_numpy().min():,.2f} .. '
          f'${lmp.to_numpy().max():,.2f} /MWh')
    print(f'  binding lines {int((util > 0.999).sum())} of {len(util)}')

    mix = (m.generators_t.p.T.groupby(m.generators.carrier).sum().sum(axis=1))
    mix = mix[mix > 0.5].sort_values(ascending=False)
    print('\n  dispatch at this snapshot (MW):')
    for carrier, mw in mix.items():
        print(f'    {carrier:14s} {mw:10,.0f}  {100*mw/mix.sum():5.1f}%')

    print('\n  most-loaded corridors:')
    for ln, u in util.sort_values(ascending=False).head(4).items():
        b0, b1 = m.lines.at[ln, 'bus0'], m.lines.at[ln, 'bus1']
        print(f"    {ln:8s} {100*u:5.1f}%   "
              f"{m.buses.at[b0, 'substation']} -> "
              f"{m.buses.at[b1, 'substation']}")


> **Exercise 6.1 — the approximation you just made.** `marginal_cost_from_gencost` keeps the linear term of MATPOWER's quadratic cost curve and throws away the curvature, because a PyPSA LP takes one constant marginal cost per generator. Quantify it: compute `c1 + 2*c2*Pmax` for every unit, compare against the `c1` you actually used, and report the mean and worst-case error in $/MWh. Then say whether it could plausibly change which units are marginal — and therefore the LMPs you just reported.

> **Exercise 6.2.** Compare the installed capacity mix above against ERCOT's actual fuel mix for the year the case represents (EIA-930 or ERCOT's own fact sheet). The generators in this case are **real** — locations, capacities and fuel types come from EIA-860. Say how close it is, and where it is not.


### What to hand in for Stage 2

State **what changes going from 123 buses to 2,000 beyond "more nodes"**. Cover at least:

- **Solve time**, measured, for a single snapshot — and extrapolate to the 8,760 hours you would want.
- **Data-cleaning effort.** You have now imported two real networks. Which import took longer, and was it proportional to the size?
- **Whether the same questions are even answerable.** TX-123BT ships five years of hourly weather-driven profiles. ACTIVSg2000 ships a single operating snapshot. Say what that costs you.
- **Voltage levels.** TX-123BT is a single-voltage 345 kV backbone. ACTIVSg2000 has 500, 230, 161 and 115 kV, plus transformers. Say what the extra realism buys, and what it costs.


---

*Before class: you swapped an aggregated network for a real one. If your site sat at one of these buses, what could you now see about its price that the aggregated version hid?*

### Sources
- **TX-123BT** — Jin Lu et al., *A Synthetic Texas Backbone Power System with Climate-Dependent Spatio-Temporal Correlated Profiles*. DOI 10.6084/m9.figshare.22144616, CC BY 4.0. rpglab.github.io/resources/TX-123BT
- **Sample SCUC code** — github.com/rpglab/SCUC_Dynamic_Line_Rating
- **ACTIVSg cases** — A. B. Birchfield et al., *Grid Structural Characteristics as Validation Criteria for Synthetic Networks*, IEEE Transactions on Power Systems. Distributed with MATPOWER (github.com/MATPOWER/matpower, `data/case_ACTIVSg*.m`) and at electricgrids.engr.tamu.edu
- **PyPSA** — pypsa.readthedocs.io. DC optimal power flow, `import_from_pypower_ppc`
- FERC/NERC/Regional Entity staff, *Final Report on the February 2021 Cold Weather Outages in Texas and the South Central United States*, November 2021 — the source for the Chapter 15 material this notebook's default day connects to
